# Project-LUCID · Colab Ollama (T4)

Colab에서 Ollama를 시작하고 Cloudflare Quick Tunnel로 로컬 LUCID에 연결하는 노트북이야.
대화, 공유 화면 관찰과 클릭 대상 선택은 모두 같은 멀티모달 모델 `gemma3:12b`를 사용해.
다른 모델을 쓰려면 이미지 입력을 지원하는 모델 이름을 아래 셀과 PC의 `LUCID_OLLAMA_MODEL`에 동일하게 설정해.

## 1. 런타임 GPU 확인

In [ ]:
!nvidia-smi

## 2. Ollama 설치 및 서비스 시작

In [ ]:
import subprocess

subprocess.run(
    ["apt-get", "install", "-y", "--no-install-recommends", "zstd"],
    check=True,
)

In [ ]:
import shutil
import subprocess

if shutil.which("ollama") is None:
    install_script = "set -o pipefail && curl -fsSL https://ollama.com/install.sh | sh"
    subprocess.run(["bash", "-lc", install_script], check=True)
else:
    print("Ollama가 이미 설치되어 있어.")

subprocess.run(["ollama", "--version"], check=True)


In [ ]:
import os
import subprocess
import time
from urllib.error import URLError
from urllib.request import urlopen

OLLAMA_API = "http://127.0.0.1:11434"
_ollama_process = globals().get("_ollama_process")

def _ollama_is_ready():
    """로컬 모델 목록 API가 응답하면 True를 반환한다."""
    try:
        with urlopen(f"{OLLAMA_API}/api/tags", timeout=2) as response:
            return response.status == 200
    except (OSError, URLError, TimeoutError):
        return False

if _ollama_is_ready():
    print("Ollama가 이미 준비되어 있어.")
else:
    if _ollama_process is None or _ollama_process.poll() is not None:
        ollama_env = os.environ.copy()
        ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
        _ollama_log = open("/content/ollama.log", "a", encoding="utf-8")
        _ollama_process = subprocess.Popen(
            ["ollama", "serve"],
            env=ollama_env,
            stdout=_ollama_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        globals()["_ollama_process"] = _ollama_process
        globals()["_ollama_log"] = _ollama_log

    deadline = time.monotonic() + 60
    while time.monotonic() < deadline:
        if _ollama_is_ready():
            print("Ollama API 준비 완료: http://127.0.0.1:11434")
            break
        if _ollama_process.poll() is not None:
            break
        time.sleep(1)
    else:
        raise TimeoutError("Ollama가 60초 안에 준비되지 않았어. /content/ollama.log를 확인해줘.")

    if not _ollama_is_ready():
        log_path = "/content/ollama.log"
        try:
            with open(log_path, encoding="utf-8", errors="replace") as log_file:
                log_tail = "".join(log_file.readlines()[-30:])
        except OSError:
            log_tail = "로그 파일을 읽을 수 없어."
        raise RuntimeError(f"Ollama 시작 실패. 로그 마지막 부분:\n{log_tail}")

## 3. 공용 멀티모달 모델 설치

In [ ]:
import json
import subprocess

MODEL_NAME = "gemma3:12b"
KEEP_ALIVE = "10m"       # 마지막 요청 후 서버가 모델을 메모리에 유지할 시간
WARMUP_TIMEOUT = 180     # 모델을 불러올 때 HTTP 응답을 기다릴 초

with urlopen(f"{OLLAMA_API}/api/tags", timeout=10) as response:
    installed_names = {model["name"] for model in json.load(response).get("models", [])}

if MODEL_NAME in installed_names:
    print(f"{MODEL_NAME} 모델이 이미 설치돼 있어.")
else:
    print(f"{MODEL_NAME} 모델 다운로드")
    subprocess.run(["ollama", "pull", MODEL_NAME], check=True)


## 4. 모델 예열과 API 확인

In [ ]:
from urllib.request import Request, urlopen

warmup_request = Request(
    f"{OLLAMA_API}/api/generate",
    headers={"Content-Type": "application/json"},
    data=json.dumps({
        "model": MODEL_NAME,
        "prompt": "",
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "options": {"num_ctx": 4096},
    }).encode("utf-8"),
)
with urlopen(warmup_request, timeout=WARMUP_TIMEOUT) as response:
    warmup_result = json.load(response)
if not warmup_result.get("done"):
    raise RuntimeError(f"모델 예열 결과를 확인해 줘: {warmup_result}")
print(f"{MODEL_NAME} 예열 완료. 요청 후 메모리 유지 시간: {KEEP_ALIVE}")

## 5. Cloudflare Quick Tunnel 시작

In [ ]:
import subprocess
from pathlib import Path

cloudflared = Path("/usr/local/bin/cloudflared")
if not cloudflared.exists() or cloudflared.stat().st_size == 0:
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", str(cloudflared),
        ],
        check=True,
    )
cloudflared.chmod(cloudflared.stat().st_mode | 0o111)
subprocess.run([str(cloudflared), "--version"], check=True)

아래 셀은 터널을 계속 실행해. 출력에 표시되는 `trycloudflare.com` 주소를 LUCID의 Ollama 주소로 사용해.
Quick Tunnel 주소는 Colab 세션마다 바뀔 수 있어.

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:11434 --http-host-header localhost:11434